In [1]:
!pip install datasets

In [2]:
%load_ext autoreload
%autoreload 2

import torch
from transformers import pipeline
import os
from tqdm.notebook import tqdm
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import time
from transformers.pipelines.pt_utils import KeyDataset
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [3]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"

# First load the model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir="./models/"
)
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    cache_dir="./models/"
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
# Then create the pipeline with the loaded model and tokenizer
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Device set to use cuda:0


In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant!"},
    {"role": "user", "content": "Tell me a short joke"},
]
outputs = pipe(
    messages,
    max_new_tokens=1024,
)
print(outputs[0]["generated_text"][-1])

{'role': 'assistant', 'content': 'A man walked into a library and asked the librarian, "Do you have any books on Pavlov\'s dogs and Schrödinger\'s cat?" The librarian replied, "It rings a bell, but I\'m not sure if it\'s here or not."'}


In [5]:
ds = load_dataset("openai/gsm8k", "main", split="test")

In [6]:
# def get_answer(row: dict) -> int:
#     """Extract the numerical answer from GSM8K format."""
#     return int(row['answer'].split('\n')[-1][5:])

In [7]:
def get_answer(row: str) -> int:
    """Extract the numerical answer from GSM8K format."""
    # Remove commas from number string before converting to int
    answer_str = row.split('\n')[-1][5:].replace(',', '')
    return int(answer_str)


In [8]:
few_shot_messages = [
            {"role": "system", "content": "You are a helpful assistant!"},
            {"role": "user", "content": "Given the following problem, reason and give a final answer to the problem.\nProblem: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem."},
            {"role": "assistant", "content": "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6"},
            {"role": "user", "content": "Given the following problem, reason and give a final answer to the problem.\nProblem: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem."},
            {"role": "assistant", "content": "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The final answer is 5"},
            {"role": "user", "content": "Given the following problem, reason and give a final answer to the problem.\nProblem: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem."},
            {"role": "assistant", "content": "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The final answer is 39"},
            {"role": "user", "content": "Given the following problem, reason and give a final answer to the problem.\nProblem: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem."},
            {"role": "assistant", "content": "Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The final answer is 8"},
            {"role": "user", "content": "Given the following problem, reason and give a final answer to the problem.\nProblem: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem."},
            {"role": "assistant", "content": "Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The final answer is 9"},
            {"role": "user", "content": "Given the following problem, reason and give a final answer to the problem.\nProblem: There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem."},
            {"role": "assistant", "content": "There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The final answer is 29"},
            {"role": "user", "content": "Given the following problem, reason and give a final answer to the problem.\nProblem: Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem."},
            {"role": "assistant", "content": "Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The final answer is 33"},
            {"role": "user", "content": "Given the following problem, reason and give a final answer to the problem.\nProblem: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem."},
            {"role": "assistant", "content": "Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The final answer is 8"},
        ]

In [9]:
def parse_answer(output: str) -> int:
    # Look for "The final answer is X" pattern
    final_answer_match = re.search(r'The final answer is (\d+)', output)
    if final_answer_match:
        return int(final_answer_match.group(1))
    print(f"Fallback: {output}")
    # Fallback: find all numbers and take the last one
    matches = re.findall(r'-?\d*\.?\d+', output)
    return int(float(matches[-1])) if matches else None

In [10]:
def evaluate_gsm8k(pipeline, dataset, num_samples=None):
    """
    Evaluate model on GSM8K dataset using batched processing.
    Args:
        pipeline: HF pipeline for text generation
        dataset: GSM8K dataset
        num_samples: Optional number of samples to evaluate (for testing)
    """
    correct = 0
    total = 0
    correct_answers = [get_answer(row['answer']) for row in dataset]
    predicted_answers = []
    for question in tqdm(KeyDataset(dataset, "question")):
        start_time = time.time()
        question_str = f"Given the following problem, reason and give a final answer to the problem.\nProblem: {question}\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem."
        messages = few_shot_messages.copy() + [{"role": "user", "content": question_str}]
        outputs = pipeline(
            messages,
            max_new_tokens=1024,
            do_sample=True,
        )
        predicted_answer = outputs[0]["generated_text"][-1]['content']
        predicted_answers.append(predicted_answer)
        # print(f"Time taken: {time.time() - start_time:.2f} seconds")
    predicted_answers = [parse_answer(answer) for answer in predicted_answers]
    for i, (predicted_answer, correct_answer) in enumerate(zip(predicted_answers, correct_answers)):
        is_correct = abs(predicted_answer - correct_answer) < 0.0001
        correct += int(is_correct)
        total += 1
        if i % 10 == 0:
            print(f"Current accuracy: {100 * correct / total:.2f}% ({correct}/{total})")
    final_accuracy = 100 * correct / total
    return {
        "accuracy": final_accuracy,
        "correct": correct,
        "total": total
    }


In [11]:
def evaluate_gsm8k_batched(pipeline, dataset, batch_size=12, num_samples=None):
    """
    Evaluate model on GSM8K dataset using batched processing.
    Args:
        pipeline: HF pipeline for text generation
        dataset: GSM8K dataset
        batch_size: Number of samples to process simultaneously
        num_samples: Optional number of samples to evaluate (for testing)
    """
    correct = 0
    total = 0

    # Prepare dataset
    if num_samples is not None:
        dataset = dataset.select(range(num_samples))

    # Get all correct answers
    correct_answers = [get_answer(row['answer']) for row in dataset]
    predicted_answers = []

    # Process in batches
    for i in tqdm(range(0, len(dataset), batch_size)):
        batch_questions = []
        batch_end = min(i + batch_size, len(dataset))

        # Prepare batch of questions
        for j in range(i, batch_end):
            question = dataset[j]['question']
            question_str = (
                f"Given the following problem, reason and give a final answer to the problem.\n"
                f"Problem: {question}\n"
                f"Your response should end with \"The final answer is [answer]\" "
                f"where [answer] is the response to the problem."
            )
            messages = few_shot_messages.copy() + [{"role": "user", "content": question_str}]
            batch_questions.append(messages)

        # Process batch
        start_time = time.time()
        outputs = pipeline(
            batch_questions,
            max_new_tokens=1024,
            do_sample=True,
            batch_size=batch_size
        )

        # Parse answers from batch
        for output in outputs:
            predicted_answers.append(output[0]["generated_text"][-1]['content'])

    predicted_answer = [parse_answer(answer) for answer in predicted_answers]
    for i, (predicted_answer, correct_answer) in enumerate(zip(predicted_answers, correct_answers)):
        is_correct = abs(predicted_answer - correct_answer) < 0.0001
        correct += int(is_correct)
        total += 1

    final_accuracy = 100 * correct / total
    return {
        "accuracy": final_accuracy,
        "correct": correct,
        "total": total,
        "predictions": predicted_answers
    }

In [12]:
# results = evaluate_gsm8k_batched(pipe, ds, batch_size=12)  # Adjust batch_size based on your GPU memory
# print(f"\nFinal Results:")
# print(f"Accuracy (EM@1): {results['accuracy']:.2f}%")
# print(f"Correct: {results['correct']}/{results['total']}")

In [ ]:
results = evaluate_gsm8k(pipe, ds)  # Adjust batch_size based on your GPU memory
print(f"\nFinal Results:")
print(f"Accuracy (EM@1): {results['accuracy']:.2f}%")
print(f"Correct: {results['correct']}/{results['total']}")

  0%|          | 0/1319 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
